# Caracal Base 3B · Continued Pretrain · Kaggle T4

Notebook template plug-and-play para uma sessao de revezamento Kaggle.

**Antes de comecar:**
1. Verificar SCHEDULE.md no repo · qual slot voce esta?
2. Pegar revision do checkpoint anterior (step-XXXX)
3. Calcular revision de saida (step-YYYY = XXXX + 5500)

**Vai rodar:**
- Auth HF + W&B
- Pull checkpoint anterior do HF Hub
- 5500 passos de continued pretrain (~9 horas T4)
- Push final novo revision pro HF Hub
- Atualizar SCHEDULE.md (manualmente apos)

## 1 · Configurar este slot

In [ ]:
# EDITAR ESTAS LINHAS POR SLOT
SESSION_NUMBER = 1  # qual sessao no SCHEDULE.md
RESUME_REVISION = None  # None = comeca do zero (so slot 1). Senao: 'step-5500' etc
OUTPUT_REVISION = 'step-5500'  # revision do checkpoint que voce vai produzir
STEPS_TO_RUN = 5500
FOUNDER_HANDLE = 'PAMF2'  # seu handle GitHub


## 2 · Setup Kaggle environment

In [ ]:
# Verificar GPU
!nvidia-smi --query-gpu=name,memory.free --format=csv,noheader

# Clone repo
!git clone https://github.com/iterate-labs-ai/caracal-1.git
%cd caracal-1
!git checkout dev

# Install deps (Unsloth + TRL + accelerate)
!pip install --quiet 'unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install --quiet trl peft bitsandbytes datasets transformers accelerate
!pip install --quiet huggingface_hub wandb pyyaml sentence-transformers

## 3 · Auth HuggingFace + W&B

In [ ]:
# Use Kaggle Secrets (Add-ons -> Secrets) para configurar:
# - HF_TOKEN (token write huggingface.co/settings/tokens)
# - WANDB_API_KEY (de wandb.ai/authorize)

from kaggle_secrets import UserSecretsClient
import os
secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')

!huggingface-cli login --token $HF_TOKEN --add-to-git-credential
!wandb login $WANDB_API_KEY

print('auth OK')

## 4 · Pre-flight check

In [ ]:
!bash scripts/preflight.sh

## 5 · Rodar continued pretrain (5500 passos, ~9h)

In [ ]:
import subprocess, sys

resume_arg = []
if RESUME_REVISION:
    resume_arg = ['--resume-from', f'huggingface://iterate-labs/caracal-base-pretrain@{RESUME_REVISION}']

cmd = [
    sys.executable, 'train/continued_pretrain.py',
    '--config', 'train/configs/caracal_base_3b.yaml',
    *resume_arg,
    '--steps-to-run', str(STEPS_TO_RUN),
    '--output', './ckpt-out',
    '--hf-revision-out', OUTPUT_REVISION,
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

## 6 · Verificar push e atualizar SCHEDULE.md manualmente

In [ ]:
# Listar revisions atuais no HF Hub
from huggingface_hub import HfApi
api = HfApi()
refs = api.list_repo_refs('iterate-labs/caracal-base-pretrain')
for ref in refs.branches[-10:]:
    print(f'  {ref.name} -> {ref.target_commit[:8]}')

# Manual: editar SCHEDULE.md pra trocar pending -> done com revision
# Manual: sinalizar Discord #caracal-relay 'sessao N completa, step-YYYY pushed'

print(f'\nSession {SESSION_NUMBER} done. Revision out: {OUTPUT_REVISION}')
print('Proximos passos manuais:')
print('1. PR pequena no caracal-1: editar SCHEDULE.md trocando pending -> done')
print('2. Sinalizar Discord #caracal-relay')
print('3. Proximo no relay resume com:')
print(f'   RESUME_REVISION = \'{OUTPUT_REVISION}\'')